In [2]:
import cv2
import mediapipe as mp
import numpy as np
from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Print versions of imports
print("Package Versions:")
print(f"OpenCV (cv2): {cv2.__version__}")
print(f"MediaPipe: {mp.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"IPython: {__import__('IPython').__version__}")

# For mediapipe.tasks, we can also check the version through mediapipe
print(f"MediaPipe Tasks: {mp.__version__}")  # Uses same version as main mediapipe

Package Versions:
OpenCV (cv2): 4.13.0
MediaPipe: 0.10.33
NumPy: 2.4.4
Matplotlib: 3.10.8
IPython: 9.12.0
MediaPipe Tasks: 0.10.33


In [3]:
import cv2
import mediapipe as mp
import numpy as np
from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [4]:
def create_overlay_animation(video_path, model_path):

    base_options = python.BaseOptions(model_asset_path=model_path)

    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    JOINTS = [
        0,   # nose/head
        11, 12,
        13, 14,
        15, 16,
        23, 24,
        25, 26,
        27, 28
    ]

    CONNECTIONS = [
        (0,1), (0,2),
        (1,3), (3,5),
        (2,4), (4,6),
        (1,2),
        (1,7), (2,8),
        (7,8),
        (7,9), (9,11),
        (8,10), (10,12)
    ]

    frames = []

    with vision.PoseLandmarker.create_from_options(options) as landmarker:

        cap = cv2.VideoCapture(video_path)

        fps = cap.get(cv2.CAP_PROP_FPS)

        frame_idx = 0

        while cap.isOpened():

            ret, frame = cap.read()

            if not ret:
                break

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=image_rgb
            )

            timestamp_ms = int((frame_idx / fps) * 1000)

            results = landmarker.detect_for_video(
                mp_image,
                timestamp_ms
            )

            output_frame = image_rgb.copy()

            if results.pose_landmarks:

                landmarks = results.pose_landmarks[0]

                points = []

                h, w, _ = output_frame.shape

                for idx in JOINTS:

                    lm = landmarks[idx]

                    x = int(lm.x * w)
                    y = int(lm.y * h)

                    points.append((x, y))

                    cv2.circle(
                        output_frame,
                        (x, y),
                        8,
                        (255, 0, 0),
                        -1
                    )

                for start, end in CONNECTIONS:

                    cv2.line(
                        output_frame,
                        points[start],
                        points[end],
                        (0,255,0),
                        2
                    )

            frames.append(output_frame)

            frame_idx += 1

        cap.release()

    fig = plt.figure(figsize=(8,6))

    img_plot = plt.imshow(frames[0])

    plt.axis("off")

    def update(i):

        img_plot.set_data(frames[i])

        return [img_plot]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=50,
        blit=True
    )

    plt.close(fig)

    return anim

In [10]:
video_path = "../../../all_videos/A121.avi"
#video_path = "../../../C25.mov"

model_path = "../data/pose_landmarker.task"



anim = create_overlay_animation(
    video_path,
    model_path
)

HTML(anim.to_html5_video())

I0000 00:00:1779629088.089326 4904222 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1779629088.147237 4904229 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779629088.166152 4904225 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
